In [13]:
import os 
import datetime
import pandas as pd
from time import sleep
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

In [2]:
load_dotenv(override=True)
API_KEY = os.environ['GEMMA_API_KEY']

In [19]:
SYSTEM_PROMPT = """
    You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
    Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
    While generating the answer make sure that:
    -Try to approximate the situation as close as you can.
    -Answer only the topic, do not add extra discussion. 
    -Do not format the answer.  
    Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

provider = GoogleProvider(api_key=API_KEY)
model = GoogleModel("gemini-2.0-flash", provider=provider)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT, output_type=str)

In [7]:
base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"

In [21]:
start_date = datetime.date(2014, 1, 1)
end_date = datetime.date(2025, 8, 10)

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    print(f"Processing for date: {current_date}")

    date_str = current_date.strftime("%Y%m%d")
    file_name = f"{date_str}_gold_filtered.csv"
    full_path = os.path.join(base_path, file_name)

    try:
        df = pd.read_csv(full_path)

        url_bulk = ""
        for url in df['SOURCEURL']:
            url_bulk = url_bulk + f" {url}"
        
        answer = await agent.run(url_bulk)
        text = answer.output

        results.append({
            "date": date_str,
            "text": text
        })
        
    except FileNotFoundError:
        print(f"File not found: {full_path}")
        missing_dates.append(current_date)
    except pd.errors.EmptyDataError:
        print(f"Empty CSV: {full_path}")
        missing_dates.append(current_date)
    except Exception as e:
        print(f"Error processing {full_path}: {e}")
        missing_dates.append(current_date)
    
    sleep(3)
    current_date += datetime.timedelta(days=1)


df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data.csv"
df.to_csv(path, index=False)

print(f"============= MISSING DATA FOR TOTAL OF {len(missing_dates)} DATES : =============")
print(missing_dates)


Processing for date: 2014-01-01
Error processing C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered\20140101_gold_filtered.csv: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.0-flash'}, 'quotaValue': '200'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '48s'}]}}

CancelledError: 